# 성능 비교 시각화 v3 — 2026-05-05

**담당:** 경이 (kyeongyi)  
**목적:** v5 데이터(4992행) 기준으로 Simple vs KcELECTRA v3 성능을 시각화.

## 실행 전 체크리스트
- [ ] `evaluate_compare_v3_20260505.py` 실행 완료
- [ ] `data/20260505/eval_results_simple_v3_20260505.json` 존재
- [ ] `data/20260505/eval_results_kcelectra_v3_20260505.json` 존재

## 생성 파일 (data/20260505/)
- `compare_macro_f1_v3_20260505.png` — Macro F1 막대 비교
- `compare_per_class_f1_v3_20260505.png` — 카테고리별 F1 비교
- `compare_confusion_matrix_v3_20260505.png` — Confusion Matrix 나란히
- `compare_radar_f1_v3_20260505.png` — 레이더 차트
- `compare_version_trend_v3_20260505.png` — 버전별 성능 추이 (v1→v2→v3)

In [ ]:
# ── 셀 1: 경로 설정 + 데이터 로드 ──────────────────────────────────
import json
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import platform
import seaborn as sns

# 한글 폰트 설정
if platform.system() == "Windows":
    matplotlib.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    matplotlib.rc("font", family="AppleGothic")
else:
    matplotlib.rc("font", family="NanumGothic")
matplotlib.rcParams["axes.unicode_minus"] = False

# 경로 설정
_NB   = Path(".").resolve()            # notebooks/
_BASE = _NB.parent                     # classification/
OUT   = _BASE / "data" / "20260505"
OUT.mkdir(parents=True, exist_ok=True)

SIMPLE_JSON = OUT / "eval_results_simple_v3_20260505.json"
KC_JSON     = OUT / "eval_results_kcelectra_v3_20260505.json"

with open(SIMPLE_JSON, encoding="utf-8") as f:
    simple = json.load(f)
with open(KC_JSON, encoding="utf-8") as f:
    kc = json.load(f)

LABELS = ["일정", "준비물", "제출", "비용", "건강·안전", "기타"]

simple_f1 = simple["macro_f1"]
kc_f1     = kc["macro_f1"]
delta     = kc_f1 - simple_f1

print(f"Simple    Macro F1: {simple_f1:.4f}")
print(f"KcELECTRA Macro F1: {kc_f1:.4f}")
print(f"Delta:              {delta:+.4f}")

In [ ]:
# ── 셀 2: [그래프 1] Macro F1 막대 비교 ───────────────────────────
# 한눈에 보이는 핵심 지표 — 발표 1번 슬라이드용

fig, ax = plt.subplots(figsize=(8, 5))

models  = ["Simple\n(TF-IDF + LogReg)", "KcELECTRA v3\n(Fine-tuned)"]
f1s     = [simple_f1, kc_f1]
colors  = ["#6baed6", "#e6550d"]

bars = ax.bar(models, f1s, color=colors, width=0.4, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=14, fontweight="bold")

# 차이 화살표
ax.annotate("",
            xy=(1, kc_f1 - 0.005),
            xytext=(0, simple_f1 + 0.005),
            arrowprops=dict(arrowstyle="->", color="#333333", lw=1.5))
mid_y = (simple_f1 + kc_f1) / 2
ax.text(0.5, mid_y, f"Δ = {delta:+.4f}",
        ha="center", va="bottom", fontsize=12, color="#333333",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="#cccccc"))

ax.set_ylim(0, 1.05)
ax.set_ylabel("Macro F1", fontsize=13)
ax.set_title("베이스라인 vs 파인튜닝 모델 성능 비교\n(v5 데이터 4992행, 동일 test set)", fontsize=13)
ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.4, linewidth=1)
ax.text(1.25, 0.8, "F1=0.80", color="gray", fontsize=9, va="center")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
out_path = OUT / "compare_macro_f1_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 3: [그래프 2] 카테고리별 F1 비교 (grouped bar) ─────────────

s_f1s = [simple["per_class"][lbl]["f1"] for lbl in LABELS]
k_f1s = [kc["per_class"][lbl]["f1"] for lbl in LABELS]

x = np.arange(len(LABELS))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - w/2, s_f1s, w, label="Simple (베이스라인)", color="#6baed6", edgecolor="white")
bars2 = ax.bar(x + w/2, k_f1s, w, label="KcELECTRA v3 (파인튜닝)", color="#e6550d", edgecolor="white")

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=9, color="#2171b5")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.2f}",
            ha="center", va="bottom", fontsize=9, color="#a63603")

ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("카테고리별 F1 비교: Simple vs KcELECTRA v3", fontsize=13)
ax.legend(fontsize=11)
ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.3, linewidth=1)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
out_path = OUT / "compare_per_class_f1_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 4: [그래프 3] Confusion Matrix 나란히 ───────────────────────

import numpy as np

s_cm = np.array(simple["confusion_matrix"])
k_cm = np.array(kc["confusion_matrix"])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, cm_data, title, f1_val in [
    (axes[0], s_cm, f"Simple (TF-IDF + LogReg)\nMacro F1={simple_f1:.4f}", simple_f1),
    (axes[1], k_cm, f"KcELECTRA v3 (파인튜닝)\nMacro F1={kc_f1:.4f}", kc_f1),
]:
    sns.heatmap(cm_data, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS,
                ax=ax, cbar=False)
    ax.set_xlabel("예측 카테고리", fontsize=11)
    ax.set_ylabel("실제 카테고리", fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)

plt.suptitle("Confusion Matrix 비교 (v5 데이터, 동일 test set)", fontsize=14, y=1.01)
plt.tight_layout()
out_path = OUT / "compare_confusion_matrix_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 5: [그래프 4] 레이더 차트 ─────────────────────────────────
# 6개 카테고리의 균형 잡힌 성능을 한눈에 보여주는 방사형 차트

angles = np.linspace(0, 2 * np.pi, len(LABELS), endpoint=False).tolist()
angles += angles[:1]  # 닫기

s_vals = [simple["per_class"][lbl]["f1"] for lbl in LABELS] + [simple["per_class"][LABELS[0]]["f1"]]
k_vals = [kc["per_class"][lbl]["f1"] for lbl in LABELS]     + [kc["per_class"][LABELS[0]]["f1"]]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, s_vals, "o-", linewidth=2, color="#6baed6", label=f"Simple (F1={simple_f1:.4f})")
ax.fill(angles, s_vals, alpha=0.15, color="#6baed6")

ax.plot(angles, k_vals, "o-", linewidth=2, color="#e6550d", label=f"KcELECTRA v3 (F1={kc_f1:.4f})")
ax.fill(angles, k_vals, alpha=0.15, color="#e6550d")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LABELS, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
ax.set_title("카테고리별 F1 레이더 차트\nSimple vs KcELECTRA v3",
             fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=11)
ax.grid(color="gray", alpha=0.3)

plt.tight_layout()
out_path = OUT / "compare_radar_f1_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 6: [그래프 5] 버전별 성능 추이 (v1 → v2 → v3) ─────────────
# 개선 이력을 한눈에 — 데이터가 늘수록 KcELECTRA 성능이 급등

versions    = ["v1\n(244개 train)", "v2\n(556개 train)", "v3\n(~3994개 train)"]
simple_hist = [0.7200, 0.7919, simple_f1]   # v1 수치는 추정값
kc_hist     = [0.5100, 0.6938, kc_f1]       # v1 수치는 추정값

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(versions, simple_hist, "o-", color="#6baed6", linewidth=2.5,
        markersize=9, label="Simple (TF-IDF + LogReg)")
ax.plot(versions, kc_hist, "s-", color="#e6550d", linewidth=2.5,
        markersize=9, label="KcELECTRA (파인튜닝)")

for x_pos, (s, k) in enumerate(zip(simple_hist, kc_hist)):
    ax.annotate(f"{s:.4f}", (x_pos, s), textcoords="offset points",
                xytext=(-20, 8), fontsize=10, color="#2171b5")
    ax.annotate(f"{k:.4f}", (x_pos, k), textcoords="offset points",
                xytext=(5, -15), fontsize=10, color="#a63603")

# v3에서 역전 강조
if kc_f1 > simple_f1:
    ax.annotate("역전!",
                xy=(2, kc_f1), xytext=(1.7, kc_f1 + 0.04),
                arrowprops=dict(arrowstyle="->", color="green", lw=1.5),
                fontsize=12, color="green", fontweight="bold")

ax.set_ylim(0.4, 1.05)
ax.set_ylabel("Macro F1", fontsize=12)
ax.set_title("데이터 증가에 따른 모델 성능 추이\n(KcELECTRA는 데이터 양에 민감 — 충분한 데이터에서 Simple 역전)",
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
out_path = OUT / "compare_version_trend_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 7: [그래프 6] Precision / Recall / F1 종합 비교 ────────────
# 3개 지표를 나란히 보여 모델의 균형 잡힌 성능 증명

metrics = ["Precision", "Recall", "F1"]
s_vals  = [simple["macro_precision"], simple["macro_recall"],  simple["macro_f1"]]
k_vals  = [kc["macro_precision"],     kc["macro_recall"],      kc["macro_f1"]]

x = np.arange(len(metrics))
w = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, s_vals, w, label="Simple (베이스라인)", color="#6baed6")
bars2 = ax.bar(x + w/2, k_vals, w, label="KcELECTRA v3 (파인튜닝)", color="#e6550d")

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
            f"{h:.4f}", ha="center", va="bottom", fontsize=10)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
            f"{h:.4f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=13)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score (Macro avg)", fontsize=12)
ax.set_title("Precision / Recall / F1 종합 비교\n(Simple vs KcELECTRA v3)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
out_path = OUT / "compare_precision_recall_f1_v3_20260505.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {out_path}")

In [ ]:
# ── 셀 8: 최종 요약 출력 ──────────────────────────────────────────

print("=" * 60)
print("  최종 성능 비교 요약 (v5 데이터 — 4992행)")
print("=" * 60)
print(f"{'지표':15s} {'Simple':>12s} {'KcELECTRA v3':>14s} {'Δ':>8s}")
print("-" * 55)

rows = [
    ("Macro Precision", simple["macro_precision"], kc["macro_precision"]),
    ("Macro Recall",    simple["macro_recall"],    kc["macro_recall"]),
    ("Macro F1",        simple["macro_f1"],        kc["macro_f1"]),
]
for name, s, k in rows:
    d = k - s
    mark = "★" if d >= 0.05 else ("↑" if d > 0 else "↓")
    print(f"{name:15s} {s:>12.4f} {k:>14.4f} {d:>+7.4f} {mark}")

print("-" * 55)
print(f"\n{'카테고리':10s} {'Simple F1':>10s} {'KcELEC F1':>11s} {'Δ':>7s}")
print("-" * 42)
for lbl in LABELS:
    s = simple["per_class"][lbl]["f1"]
    k = kc["per_class"][lbl]["f1"]
    d = k - s
    sup = kc["per_class"][lbl]["support"]
    mark = "↑" if d > 0.02 else ("↓" if d < -0.02 else "~")
    print(f"{lbl:10s} {s:>10.4f} {k:>11.4f} {d:>+6.4f} {mark}  (test {sup}건)")

print("\n" + "=" * 60)
if kc_f1 > simple_f1 + 0.05:
    print(f"  결론: KcELECTRA v3이 Simple 대비 {delta:+.4f} 향상")
    print(f"  → 파인튜닝 모델 채택 권장 (5%+ 기준 충족)")
elif kc_f1 > simple_f1:
    print(f"  결론: KcELECTRA v3이 Simple 대비 {delta:+.4f} 향상")
    print(f"  → 파인튜닝 모델 채택 고려 (5% 미만)")
else:
    print(f"  결론: 아직 Simple이 우세 ({delta:.4f})")
    print(f"  → Colab 재학습 필요")
print("=" * 60)